# GitHub AI-Agent Star Velocity

This notebook uses real GitHub API data to track developer attention for AI-agent and developer-tool repositories.

Stars are an attention proxy, not production adoption. No synthetic fallback is used.


In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

# Prefer the checkout when this notebook is run inside the repository.
repo_root = Path.cwd()
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
    if (candidate / "src" / "detime").exists():
        repo_root = candidate
        break
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))

from examples.hot_trends.data import (
    HotTrendDataError,
    append_real_snapshot,
    build_arxiv_monthly_counts,
    fetch_coingecko_market_chart,
    fetch_defillama_stablecoin_chains,
    fetch_github_repo_metadata,
    fetch_github_stargazers,
    fetch_huggingface_models,
    fetch_wikipedia_pageviews,
    source_audit_table,
)
from examples.hot_trends.decomposition import (
    component_summary,
    decompose_table,
    editorial_priority,
    residual_event_table,
)
from examples.hot_trends.scoring import article_language_guardrails

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

CACHE_DIR = repo_root / "examples" / "hot_trends" / "cache"
OUTPUT_DIR = repo_root / "examples" / "hot_trends" / "outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_table(df, name):
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"saved: {path.relative_to(repo_root)}")
    return path


## 1. Select repositories

Use a small list for unauthenticated runs. Set `GITHUB_TOKEN` for higher rate limits.


In [ ]:
repos = [
    "langchain-ai/langchain",
    "microsoft/autogen",
    "crewAIInc/crewAI",
    "browser-use/browser-use",
    "modelcontextprotocol/servers",
]
token = os.environ.get("GITHUB_TOKEN")
pd.DataFrame({"repo": repos})


## 2. Fetch repository metadata


In [ ]:
metadata_rows = []
for repo in repos:
    meta = fetch_github_repo_metadata(repo, token=token)
    metadata_rows.append({
        "repo": repo,
        "stars": meta.get("stargazers_count"),
        "forks": meta.get("forks_count"),
        "open_issues": meta.get("open_issues_count"),
        "pushed_at": meta.get("pushed_at"),
        "source": "GitHub REST API",
    })
metadata = pd.DataFrame(metadata_rows).sort_values("stars", ascending=False)
metadata


## 3. Fetch stargazer timestamps

The endpoint returns the latest pages by default. For full histories, paginate further or use a scheduled snapshot system.


In [ ]:
star_rows = []
for repo in repos:
    sg = fetch_github_stargazers(repo, pages=3, token=token)
    star_rows.append(sg)
stars = pd.concat(star_rows, ignore_index=True)
stars.head(20)


## 4. Convert stargazer timestamps to daily velocity


In [ ]:
stars["date"] = pd.to_datetime(stars["starred_at"]).dt.date.astype(str)
daily = stars.groupby(["repo", "date"]).size().reset_index(name="count")
daily.head(20)


## 5. Decompose star velocity if enough days exist


In [ ]:
coverage = source_audit_table(daily, value_col="count", entity_col="repo", time_col="date")
ready = coverage.loc[coverage["observations"] >= 14, "series"].tolist()
if ready:
    components = decompose_table(daily[daily["repo"].isin(ready)], entity_col="repo", time_col="date", value_col="count", method="MA_BASELINE", period=7, trend_window=5, transform="log1p")
    summary = editorial_priority(component_summary(components, entity_col="repo", time_col="date"), entity_col="repo")
    events = residual_event_table(components, entity_col="repo", time_col="date", top_n=20)
else:
    components = pd.DataFrame()
    summary = pd.DataFrame([{"status": "not_enough_real_stargazer_days", "required": "increase pages or run repeated snapshots"}])
    events = pd.DataFrame()
coverage


In [ ]:
summary


## 6. Article-safe interpretation


In [ ]:
guardrails = article_language_guardrails()
guardrails


In [ ]:
save_table(metadata, "04_github_repo_metadata")
save_table(coverage, "04_github_stargazer_coverage")
save_table(daily, "04_github_star_velocity_daily")
save_table(summary, "04_github_decomposition_or_collection_status")
if not events.empty:
    save_table(events, "04_github_residual_events")
save_table(guardrails, "04_github_guardrails")
